# 4.6 Structured Output in Your App: Concepts Notebook

**Generative AI Foundations · Module 4**

| # | Concept |
|---|---|
| 1 | Text vs data |
| 2 | A schema is a form |
| 3 | Getting an object back |
| 4 | Missing information: `None` |
| 5 | Many messages at once |
| 6 | When the form comes back wrong |
| 7 | Letting code decide: a router |
| 8 | Summarizing a conversation |

**Model:** `gemma4:e2b` (`ollama pull gemma4:e2b`).

## 0. Setup

In [ ]:
%pip install -q "langchain[ollama]"

In [ ]:
from typing import Literal

from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field, ValidationError

llm = init_chat_model("ollama:gemma4:e2b", temperature=0)
print("Ready")

---
## 1. Text vs Data

> **Mental Model: The Form at the Counter**
> A customer explains a problem in their own words. The clerk fills in a **form**. The office works with the form, not the story.

A normal call gives **text**. Your code can't easily use it:

In [ ]:
message = "I was charged twice for order #4821, please fix this today!"

reply = llm.invoke(f"What category and order number is this? {message}")
print(reply.text)

Which part is the category? Where is the order number? Your code would have to guess. Structured output fixes this.

---
## 2. A Schema Is a Form

A Pydantic class describes the boxes on the form:

| Box | Type | Meaning |
|---|---|---|
| `category` | one of a fixed list | the topic |
| `priority` | `low` / `medium` / `high` | how urgent |
| `order_number` | text or nothing | the order, if mentioned |

In [ ]:
class TicketInfo(BaseModel):
    """Key facts from one customer message."""

    category: Literal["billing", "delivery", "technical", "account", "other"] = Field(
        description="Main topic. If money is involved, use billing."
    )
    priority: Literal["low", "medium", "high"]
    order_number: str | None = Field(default=None, description="Order number such as #4821, or null")


print(list(TicketInfo.model_json_schema()["properties"]))

This JSON Schema is what LangChain sends to the model, so the model knows exactly which boxes to fill.

---
## 3. Getting an Object Back

```text
message --> llm.with_structured_output(TicketInfo) --> TicketInfo(category=..., priority=..., order_number=...)
```

In [ ]:
extractor = llm.with_structured_output(TicketInfo)

info = extractor.invoke(message)

print(type(info).__name__)
print("category:    ", info.category)
print("priority:    ", info.priority)
print("order_number:", info.order_number)

Now your code can use the fields directly:

In [ ]:
if info.priority == "high":
    print(f"Alert: urgent {info.category} problem on order {info.order_number}")

print(info.model_dump())   # as a normal dictionary

---
## 4. Missing Information: `None`

When a message has no order number, the model should say **nothing**, not invent one. `str | None` with `default=None` gives it a safe "not mentioned" answer.

In [ ]:
info = extractor.invoke("How do I change my email address?")
print("order_number:", info.order_number)

---
## 5. Many Messages at Once

`batch` (Lesson 4.2) works with structured output too. You get a **list of objects**, in the same order.

In [ ]:
messages = [
    "I was charged twice for order #4821!",
    "My parcel hasn't arrived. Order #5521.",
    "How do I change my email address?",
]

for text, info in zip(messages, extractor.batch(messages)):
    print(f"{info.category:<10} {info.priority:<7} {str(info.order_number):<6} <- {text}")

---
## 6. When the Form Comes Back Wrong

> **Mental Model: The Half-Filled Form**
> Sometimes a form comes back with a wrong value or a box left empty. A good office has a rule for that, instead of filing it as correct.

Pydantic **checks** every value. Here is what happens with a value that is not allowed (no model needed):

In [ ]:
try:
    TicketInfo.model_validate({"category": "food", "priority": "high"})
except ValidationError as error:
    print("Rejected:", error.errors()[0]["msg"])

With a model, use `include_raw=True` so a bad answer never crashes your app. You get three things back:

| Key | Contains |
|---|---|
| `parsed` | The object, or `None` if it failed |
| `parsing_error` | The error, or `None` |
| `raw` | The model's original message |

In [ ]:
safe_extractor = llm.with_structured_output(TicketInfo, include_raw=True)

result = safe_extractor.invoke("My parcel is late. Order #5521.")

print("keys:         ", list(result))
print("parsing_error:", result["parsing_error"])
print("parsed:       ", result["parsed"])

if result["parsing_error"] is not None:
    print("Fallback: ask the user to rephrase, or pass to a human.")

---
## 7. Letting Code Decide: a Router

One small structured call decides the path. Then **normal Python** (`if` / `elif`) does the rest.

```text
message -> Route(intent=...) -> "tools"   -> run tools
                             -> "handoff" -> a human will reply
                             -> "chat"    -> normal answer
```

In [ ]:
class Route(BaseModel):
    intent: Literal["chat", "tools", "handoff"] = Field(
        description="tools: time, currency, or order status. handoff: refunds, payment problems, upset customers. chat: anything else."
    )


router = llm.with_structured_output(Route)

for text in ["What is the status of order #4821?", "I was charged twice, refund me!", "What is Python?"]:
    route = router.invoke(text)
    if route.intent == "handoff":
        action = "send to a human agent"
    elif route.intent == "tools":
        action = "run tools (Lesson 4.5)"
    else:
        action = "answer normally"
    print(f"{route.intent:<8} -> {action:<26} <- {text}")

The refund never reaches a normal model reply. **Code** decided, using a typed value.

---
## 8. Summarizing a Conversation

You can give a structured model a **whole list of messages**, not just one string.

In [ ]:
class ChatSummary(BaseModel):
    customer_name: str | None = Field(default=None, description="The customer's name, or null")
    topics: list[str] = Field(description="Main topics, 1 to 4 short phrases")
    needs_human: bool = Field(description="True if a human should follow up")


conversation = [
    {"role": "system", "content": "Summarize this support conversation."},
    {"role": "user", "content": "Hi, my name is Ali."},
    {"role": "assistant", "content": "Hello Ali! How can I help?"},
    {"role": "user", "content": "My card was charged twice for order #4821."},
    {"role": "assistant", "content": "Sorry! I'll pass this to our billing team."},
]

summary = llm.with_structured_output(ChatSummary).invoke(conversation)
print(summary.model_dump_json(indent=2))

---
## Summary

| Concept | Remember |
|---|---|
| Text vs data | Users read text; code needs data |
| Schema | A Pydantic class: the form with typed boxes |
| `with_structured_output(Schema)` | Returns a `Schema` object instead of text |
| `None` default | A safe way to say "not mentioned" |
| `batch` | A list of objects, same order |
| `include_raw=True` | Check `parsing_error`; always have a fallback |
| Router | A typed decision, then normal `if` / `elif` |
| Summary | Pass a whole message list; save with `model_dump_json()` |

## Practice

1. Add a `sentiment` field (`positive`, `neutral`, `negative`) to `TicketInfo` and run Section 5 again.
2. In Section 7, add a fourth intent, `"complaint"`, and decide what your code should do with it.
3. In Section 8, add an `open_issue: str | None` field and print only that field.